# Phase 5 escalation B - unfreeze the decoder (LoRA on encoder + decoder)

**You run this in Colab. Claude Code wrote it and cannot run it.**

**This departs from the proposal's "keep NLLB frozen" framing** and needs a
methodology caveat in the write-up. It adds LoRA adapters on the **encoder
AND decoder** (self-attn + the decoder's cross-attn), so the model has
actual capacity to build a pathway that could differentiate a
better-segmented source tokenisation from a worse-segmented one -- the
frozen-decoder escalation A (`local_lora.py` / `phase5-lora-encoder.ipynb`)
never gave the decoder a chance to react to the new tokenisation at all.

### The comparison: constraint ablation, not native-tokenizer baseline
This notebook trains **two conditions under an identical LoRA setup**, both
of which are encoder-embedding-swap conditions (matched vocab, matched
corpus, matched warm-start recipe) -- only the *morphology-boundary
crossing penalty* differs:

| condition | source tokenizer | embedding |
|---|---|---|
| `penalty8` | weighted MorphBPE, crossing penalty 8 @ 6,080 | warm-started `nn.Embedding(6080,1024)`, trainable |
| `bpe6080` | plain (unconstrained) BPE, same algorithm + corpus @ 6,080 | warm-started `nn.Embedding(6080,1024)`, trainable |

**The real test is `penalty8` vs `bpe6080`** -- identical adaptation budget,
identical embedding-swap mechanism, identical vocab size and training
corpus; only the morphology constraint differs. This is a tighter match
than comparing against `nllb_native` (which uses a completely different,
untouched 256K-row embedding table) -- here every part of the pipeline is
the same except the token-id assignment itself. `nllb_zeroshot` (no
training) stays in only as an untrained reference point, not a tested
condition.

Trainable for both conditions: warm-started embedding + encoder&decoder
LoRA. lm_head + `shared` stay frozen; `tie_weights()` never called after
the swap.

### How to run
1. `Runtime -> Change runtime type -> GPU`. **L4 or A100 preferred** - the
   decoder adapters + longer graph use more memory than escalation A; T4
   may need `BATCH` dropped to 4 in `meta.json` or the config cell.
2. Upload the same 6 bundle files (now with `bpe_ids` / `bpe6080` vocab
   included). Keys are tagged `-loraENCDEC`.
3. `Runtime -> Run all`. Default = **`penalty8` + `bpe6080`, 2 seeds
   each** = 4 runs + zero-shot reference. ~2-4 h per run on a T4. Resumable.
4. Download `phase5-results.json` and send it back.

**Data note:** as escalation A - Bible corpus rights UNCONFIRMED; Colab
runtime only.

**Proposal note:** this comparison (MorphBPE vs plain BPE, both adapted
into NLLB) departs from the currently-approved proposal's stated baseline
("the native NLLB-200 tokenizer"). Treat this as supporting the SOP/RQ/
hypothesis redraft under discussion, not as satisfying the proposal as
currently written, until that redraft is approved.

In [ ]:
%pip install -q -U "transformers>=4.44,<5" "peft>=0.11" "torchao>=0.16.0" sentencepiece sacremoses sacrebleu
import json, os, random, time
import numpy as np
import torch, torch.nn as nn
import transformers, peft, sacrebleu
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| torch", torch.__version__, "| sacrebleu", sacrebleu.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    BF16 = torch.cuda.is_bf16_supported()
    print("GPU:", p.name, round(p.total_memory/1024**3, 1), "GiB | bf16:", BF16)
else:
    BF16 = False
    print("WARNING: no GPU - this will be unusably slow.")
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
USE_SCALER = (DEVICE == 'cuda') and not BF16   # fp16 needs loss scaling; bf16 does not

In [ ]:
# ============ CONFIG - edit here ============
TAG = "loraENCDEC"                    # result keys:  penalty8-loraENCDEC/seed0
CONDITIONS = ["penalty8", "bpe6080"]   # constraint ablation: same algorithm+corpus+vocab, only the morphology-boundary penalty differs
SEEDS = [0, 1]
LORA_R = 16
LORA_ALPHA = 32
LORA_TARGETS = "q_proj,v_proj"       # applied to BOTH encoder and decoder attention
CFG = dict(epochs=20, patience=4, emb_lr=1e-3, lora_lr=2e-4)
# ===========================================
_proj = '|'.join(t.strip() for t in LORA_TARGETS.split(','))
# encoder self-attn + decoder self-attn + decoder cross-attn (encoder_attn)
LORA_TARGET_REGEX = rf'model\.(encoder|decoder)\.layers\.\d+\.(self_attn|encoder_attn)\.({_proj})'
print('LoRA target regex:', LORA_TARGET_REGEX)

In [ ]:
# --- load the bundle (same 6 files as the base Phase 5 notebook) ---
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

for fn in ["train.jsonl", "dev.jsonl", "test_bible.jsonl", "test_ood.jsonl", "meta.json", "vocab.json"]:
    assert os.path.exists(fn), f"missing {fn} - upload the bundle files first"

META = json.load(open("meta.json", encoding="utf-8"))
VOCAB = json.load(open("vocab.json", encoding="utf-8"))
TRAIN = read_jsonl("train.jsonl"); DEV = read_jsonl("dev.jsonl")
TEST_SETS = {"test_bible": read_jsonl("test_bible.jsonl"), "test_ood": read_jsonl("test_ood.jsonl")}
print(f"train {len(TRAIN)} | dev {len(DEV)} | test_bible {len(TEST_SETS['test_bible'])} | test_ood {len(TEST_SETS['test_ood'])}")

TGT_LANG = META["nllb"]["target_lang"]
NATIVE_SRC_LANG = META["nllb"]["native_baseline_source_lang"]
SRC_VOCAB = META["source_vocab_size_morphbpe"]        # 6080
SRC_PAD = META["source_pad_id_morphbpe"]              # 0
MODEL_NAME = META["nllb"]["model"]
HP = META["training"]["hyperparams"]
SWAP_CONDITIONS = set(META["conditions"])             # morphbpe, penalty8, unigram6080

RESULTS_PATH = "phase5-results.json"
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
def save_results():
    json.dump(results, open(RESULTS_PATH, "w"), indent=2, ensure_ascii=False)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.tgt_lang = TGT_LANG
TGT_BOS = tokenizer.convert_tokens_to_ids(TGT_LANG)
PAD = tokenizer.pad_token_id

def fresh_model():
    m = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    m.config.use_cache = False
    return m

def src_ids_for(condition, rec):
    if condition == "morphbpe":    return rec["morphbpe_ids"]
    if condition == "penalty8":    return rec["penalty8_ids"]
    if condition == "bpe6080":     return rec["bpe_ids"]
    if condition == "unigram6080": return rec["unigram_ids"]
    tokenizer.src_lang = NATIVE_SRC_LANG                  # nllb_native / nllb_zeroshot
    return tokenizer(rec["pam_text"], add_special_tokens=True)["input_ids"]

def label_ids(rec):
    return tokenizer(text_target=rec["fil_text"], add_special_tokens=True)["input_ids"]

def src_pad_for(condition):
    return SRC_PAD if condition in SWAP_CONDITIONS else PAD

def collate(batch, condition):
    src = [src_ids_for(condition, r) for r in batch]
    lab = [label_ids(r) for r in batch]
    sm = max(len(x) for x in src); lm = max(len(x) for x in lab); pad = src_pad_for(condition)
    ii = torch.full((len(batch), sm), pad, dtype=torch.long)
    am = torch.zeros((len(batch), sm), dtype=torch.long)
    lb = torch.full((len(batch), lm), -100, dtype=torch.long)
    for i,(s,l) in enumerate(zip(src, lab)):
        ii[i,:len(s)] = torch.tensor(s); am[i,:len(s)] = 1
        lb[i,:len(l)] = torch.tensor(l)
    return ii.to(DEVICE), am.to(DEVICE), lb.to(DEVICE)

In [ ]:
# --- warm-started encoder embedding (Phase 4 recipe; never tie_weights after) ---
NLLB_SPECIAL_ROW = {0: PAD, 1: tokenizer.unk_token_id, 2: tokenizer.bos_token_id, 3: tokenizer.eos_token_id}

def warm_start_embedding(condition, shared_weight):
    d = shared_weight.size(1)
    emb = nn.Embedding(SRC_VOCAB, d, padding_idx=SRC_PAD)
    strings = VOCAB[condition]
    with torch.no_grad():
        emb.weight.normal_(0.0, d ** -0.5)
        for i, s in enumerate(strings):
            if i in NLLB_SPECIAL_ROW:
                emb.weight[i] = shared_weight[NLLB_SPECIAL_ROW[i]].cpu(); continue
            sub = tokenizer(s, add_special_tokens=False)["input_ids"]
            if sub:
                emb.weight[i] = shared_weight[sub].mean(0).cpu()
        emb.weight[SRC_PAD].zero_()
    return emb

def build_model(condition, seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    m = fresh_model()
    for p in m.parameters():
        p.requires_grad_(False)
    emb = None
    if condition in SWAP_CONDITIONS:
        emb = warm_start_embedding(condition, m.model.shared.weight.detach())
        emb.weight.requires_grad_(True)
        m.model.encoder.embed_tokens = emb            # encoder ONLY
    cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                     target_modules=LORA_TARGET_REGEX, bias='none', task_type='SEQ_2_SEQ_LM')
    m = get_peft_model(m, cfg)
    base = m.get_base_model()
    if emb is not None:
        base.model.encoder.embed_tokens.weight.requires_grad_(True)
        emb = base.model.encoder.embed_tokens
    m.to(DEVICE)
    lora_params = [p for n,p in m.named_parameters() if 'lora_' in n and p.requires_grad]
    n_lora = sum(p.numel() for p in lora_params)
    n_emb = emb.weight.numel() if emb is not None else 0
    print(f'   trainable: embedding {n_emb:,} + LoRA {n_lora:,} = {n_emb + n_lora:,}')
    return m, emb, lora_params

In [ ]:
BATCH = HP['batch']; LR_EMB = CFG['emb_lr']; LR_LORA = CFG['lora_lr']
EPOCHS = CFG['epochs']; PATIENCE = CFG['patience']
GEN_KW = dict(num_beams=4, max_new_tokens=HP.get('max_new_tokens', 160), forced_bos_token_id=TGT_BOS)
scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)

def batched(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

@torch.no_grad()
def dev_loss(m, condition):
    m.eval(); tot = 0.0; nb = 0
    for chunk in batched(DEV, BATCH):
        ii, am, lb = collate(chunk, condition)
        with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=DEVICE=='cuda'):
            tot += float(m(input_ids=ii, attention_mask=am, labels=lb).loss)
        nb += 1
    return tot / nb

@torch.no_grad()
def eval_gen(m, data, condition):
    m.eval(); hyps = []; refs = [r['fil_text'] for r in data]
    for chunk in batched(data, BATCH):
        src = [src_ids_for(condition, r) for r in chunk]
        sm = max(len(x) for x in src); pad = src_pad_for(condition)
        ii = torch.full((len(chunk), sm), pad, dtype=torch.long)
        am = torch.zeros((len(chunk), sm), dtype=torch.long)
        for i,s in enumerate(src):
            ii[i,:len(s)] = torch.tensor(s); am[i,:len(s)] = 1
        with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=DEVICE=='cuda'):
            out = m.generate(input_ids=ii.to(DEVICE), attention_mask=am.to(DEVICE), **GEN_KW)
        hyps += tokenizer.batch_decode(out, skip_special_tokens=True)
    return {'bleu': round(sacrebleu.corpus_bleu(hyps, [refs]).score, 2),
            'chrf': round(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score, 2)}

def eval_all_tests(m, condition):
    return {name: eval_gen(m, data, condition) for name, data in TEST_SETS.items()}

def train_one(condition, seed):
    key = f'{condition}-{TAG}/seed{seed}'
    if key in results:
        print('skip (done):', key); return
    t0 = time.time()
    m, emb, lora_params = build_model(condition, seed)
    groups = [{'params': lora_params, 'lr': LR_LORA}]
    if emb is not None:
        groups.append({'params': [emb.weight], 'lr': LR_EMB})
    opt = torch.optim.AdamW(groups)
    order = list(range(len(TRAIN)))
    best = {n: p.detach().clone() for n,p in m.named_parameters() if p.requires_grad}
    best_loss = float('inf'); best_ep = 0; bad = 0
    for ep in range(1, EPOCHS+1):
        m.train(); random.Random(1000+seed*97+ep).shuffle(order)
        tr = 0.0; nb = 0
        for idx in batched(order, BATCH):
            ii, am, lb = collate([TRAIN[i] for i in idx], condition)
            with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=DEVICE=='cuda'):
                loss = m(input_ids=ii, attention_mask=am, labels=lb).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad()
            tr += float(loss.detach()); nb += 1
        dl = dev_loss(m, condition)
        print(f'  {key} ep{ep:02d}  train {tr/nb:.3f}  dev-loss {dl:.3f}')
        if dl < best_loss - 1e-3:
            best_loss = dl; best_ep = ep; bad = 0
            best = {n: p.detach().clone() for n,p in m.named_parameters() if p.requires_grad}
        else:
            bad += 1
            if bad >= PATIENCE:
                print('  early stop'); break
    with torch.no_grad():
        for n,p in m.named_parameters():
            if n in best: p.copy_(best[n])
    dev_score = eval_gen(m, DEV, condition)
    tests = eval_all_tests(m, condition)
    results[key] = {'condition': f'{condition}-{TAG}', 'seed': seed, 'lora': {'r': LORA_R, 'alpha': LORA_ALPHA, 'targets': LORA_TARGETS},
                    'best_epoch': best_ep, 'best_dev_loss': round(best_loss, 3),
                    'emb_lr': LR_EMB, 'lora_lr': LR_LORA, 'dev': dev_score, **tests,
                    'minutes': round((time.time()-t0)/60, 1)}
    save_results()
    print(f"  -> {key}  test_bible {tests['test_bible']}  test_ood {tests['test_ood']}  ({results[key]['minutes']} min)")
    del m; torch.cuda.empty_cache()

In [ ]:
# --- nllb_zeroshot reference (no training), once ---
if 'nllb_zeroshot' not in results:
    m = fresh_model().to(DEVICE)
    results['nllb_zeroshot'] = {'condition': 'nllb_zeroshot', 'dev': eval_gen(m, DEV, 'nllb_zeroshot'),
                               **eval_all_tests(m, 'nllb_zeroshot')}
    save_results(); del m; torch.cuda.empty_cache()
    print('nllb_zeroshot', results['nllb_zeroshot'])

for condition in CONDITIONS:
    for seed in SEEDS:
        train_one(condition, seed)

In [ ]:
# --- summary + download ---
from statistics import mean, pstdev
TEST_NAMES = list(TEST_SETS)
agg = {}
for condition in CONDITIONS:
    runs = [results[f'{condition}-{TAG}/seed{s}'] for s in SEEDS if f'{condition}-{TAG}/seed{s}' in results]
    if not runs: continue
    row = {'n_seeds': len(runs)}
    for tn in TEST_NAMES:
        tc = [r[tn]['chrf'] for r in runs]; tb = [r[tn]['bleu'] for r in runs]
        row[tn] = {'chrf_mean': round(mean(tc),2), 'chrf_std': round(pstdev(tc),2),
                   'bleu_mean': round(mean(tb),2), 'bleu_std': round(pstdev(tb),2)}
    agg[condition + '-' + TAG] = row
if 'nllb_zeroshot' in results:
    agg['nllb_zeroshot'] = {tn: results['nllb_zeroshot'][tn] for tn in TEST_NAMES}
print(json.dumps(agg, indent=2))
json.dump({'aggregate': agg, 'per_run': results,
           'meta': {'tag': TAG, 'lora': {'r': LORA_R, 'alpha': LORA_ALPHA, 'targets': LORA_TARGETS},
                    'conditions': CONDITIONS, 'seeds': SEEDS, 'cfg': CFG, 'hp': HP,
                    'all_silver': True}},
          open('phase5-results.json', 'w'), indent=2, ensure_ascii=False)
try:
    from google.colab import files; files.download('phase5-results.json')
except Exception as e:
    print('download manually:', e)

## Reading the result

**Primary comparison - `penalty8-loraENCDEC` vs `bpe6080-loraENCDEC`**
(identical adaptation budget, identical embedding-swap mechanism, identical
vocab/corpus -- only the morphology-boundary crossing penalty differs), on
`test_bible`:

- `penalty8` **clearly above** `bpe6080` (> across-seed std) -> the
  morphology constraint helps once the model can adapt end-to-end. A
  positive result for the thesis - report it *with* the caveat that this
  unfreezes the decoder, departing from the proposal's stated frozen setup.
- `penalty8` **~= or below** `bpe6080` -> the morphology constraint brings
  no downstream benefit even with a fully adaptable model; consistent with
  the encoder-only-LoRA result (escalation A), where the same two
  conditions landed within ~0.1 chrF++ of each other. The null result
  would then be robust across both adaptation budgets.

Secondary: either trained condition vs `nllb_zeroshot` (33.6) shows whether
decoder adaptation on ~3.6k pairs helps *at all* (it may hurt - catastrophic
forgetting of NLLB's Filipino fluency is a real risk).

Send `phase5-results.json` back -> `experiments/nllb_finetune_v1/reports/`.